In [1]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt


pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [2]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)

data_gdynia_long_categories = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)

In [3]:
data_gdynia

,osm_id,addr_street,addr_number,ground_floor_group,ground_floor_function,upper_floor_group,upper_floor_function,number_of_storeys,is_mixed_use,build_year_est,arch_style,condition,max_height,is_heritage,source,confidence,date_of_checking,number_of_functions
id,,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,hotel,NaN,hotel,7,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
2,890729731,Świętego Piotra,6D,NaN,mieszkaniowa wielorodzinna,NaN,mieszkaniowa wielorodzinna,5,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
3,890292040,Węglowa,22,NaN,centrum handlowe,NaN,mieszkaniowa wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
4,890729732,Świętego Piotra,6,NaN,mieszkaniowa wielorodzinna,NaN,mieszkaniowa wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
5,136885205,Tadeusza Wendy,15,NaN,centrum medyczne,NaN,centrum medyczne,6,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,2,2026-09-27,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1319,"125910065,260676399",Ignacego Krasickiego,2A,NaN,mieszkaniowa wielorodzinna,NaN,mieszkaniowa wielorodzinna,12,<NA>,1980,współczesność,2,NaN,NaN,google.com/maps,2,2026-10-22,1
1320,260676012,NaN,NaN,NaN,garaż,NaN,NaN,1,<NA>,1980,współczesność,2,NaN,NaN,google.com/maps,3,2026-10-22,1
1323,126106116,Żołnierzy I Armii Wojska Polskiego,6,NaN,mieszkaniowa jednorodzinna,NaN,mieszkaniowa jednorodzinna,3,<NA>,1930,klasycyzm,3,NaN,NaN,google.com/maps,2,2026-10-22,1


1.

In [4]:
dtab1 = (
    data_gdynia_long_categories.groupby(["addr_street", "category"])
    .size()
    .unstack(fill_value=0)
)
dtab2 = pd.pivot_table(
    data_gdynia_long_categories,
    index="addr_street",
    columns="category",
    values="function_clean",
    aggfunc="size",
    fill_value=0,
)
dtab3 = pd.crosstab(
    index=data_gdynia_long_categories.reset_index().loc[:, "addr_street"],
    columns=data_gdynia_long_categories.reset_index().loc[:, "category"],
)

In [5]:
pd.testing.assert_frame_equal(dtab1, dtab2)
pd.testing.assert_frame_equal(dtab2, dtab3)

assert (dtab1 == 0).sum(axis=None) == 443
assert dtab1.sum(axis=None) == 1124
assert dtab1.sum(axis=1).loc["Świętojańska"] == 229

2.

In [6]:
dtab21 = pd.crosstab(
    index=data_gdynia_long_categories.reset_index().loc[:, "addr_street"],
    columns=data_gdynia_long_categories.reset_index().loc[:, "category"],
    normalize="index",
    margins=True,
)
dtab22 = pd.crosstab(
    index=data_gdynia_long_categories.reset_index().loc[:, "addr_street"],
    columns=data_gdynia_long_categories.reset_index().loc[:, "category"],
    normalize="columns",
)
dtab23 = pd.crosstab(
    index=data_gdynia_long_categories.reset_index().loc[:, "addr_street"],
    columns=data_gdynia_long_categories.reset_index().loc[:, "category"],
    normalize="all",
)

In [7]:
assert np.isclose(dtab21.sum(axis=1), 1).all()
assert np.isclose(dtab22.sum(axis=0), 1).all()
assert np.isclose(dtab23.sum().sum(), 1)

In [8]:
dtab22.style.background_gradient(cmap="Blues", axis="index").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)

category,biura,brak parteru,finanse,garaże,gastronomia,handel,kult religijny,kultura i rozrywka,mieszkaniowa,noclegi,organizacje publiczne,przemysł i infrastruktura,pustostan,transport,usługi,zdrowie
addr_street,,,,,,,,,,,,,,,,
10 Lutego,,5.3%,8.7%,,4.6%,7.2%,,,0.9%,13.3%,9.8%,,6.5%,,3.1%,4.0%
3 Maja,,,2.2%,18.2%,1.7%,2.4%,,6.1%,3.6%,,3.9%,,2.2%,,3.1%,2.0%
Aleja Jana Pawła II,3.6%,,,,4.6%,1.0%,,18.2%,,,2.0%,,,10.0%,,
Aleja Marszałka Józefa Piłsudskiego,,,,,1.1%,,,,1.8%,,2.0%,,,,,1.0%
Antoniego Abrahama,,15.8%,21.7%,18.2%,14.9%,12.0%,,15.2%,3.6%,6.7%,5.9%,,10.9%,,11.3%,7.9%
Antoniego Hryniewickiego,28.6%,,,,0.6%,,,,,,,52.4%,6.5%,,,3.0%
Arkadiusza Rybickiego,,,,,1.1%,0.3%,,,,,,9.5%,6.5%,,1.3%,2.0%
Armii Krajowej,7.1%,21.1%,,9.1%,1.7%,3.1%,42.9%,,3.6%,6.7%,9.8%,,2.2%,,0.6%,3.0%
Dworcowa,,,2.2%,,0.6%,1.0%,,,,,,,,10.0%,,


3.

In [9]:
# def add_building_features(df_short,df_long):

# merge = df1.
# return df1.assign

In [10]:
PERIOD_BINS = [1800, 1920, 1940, 1960, 1990, 2030]
PERIOD_LABELS = ["before1920", "1920-39", "1940-59", "1960-89", "1990+"]
feature_names = ["high-rise", "garage", "mixed-use", "service", "residential"]


def feature_types(d, high_rise):
    return [
        d["number_of_storeys"].ge(high_rise),
        d["is_garage"],
        d["is_service"] & d["upper_floor_function"].str.contains("mieszkaniowa"),
        d["is_service"],
        d["is_residential"] | d["upper_floor_function"].str.contains("mieszkaniowa"),
    ]


def add_building_features(
    df_short, df_long, high_rise=10, bins=PERIOD_BINS, labels=PERIOD_LABELS
):

    exc_func_for_service = [
        "garaże",
        "mieszkaniowa",
        "pustostan",
        "brak parteru",
        "przemysł i infrastruktura",
    ]
    data_long_features = (
        df_long.assign(
            is_garage=lambda d: d["category"] == "garaże",
            is_residential=lambda d: d["category"] == "mieszkaniowa",
            is_vacant=lambda d: d["category"] == "pustostan",
            is_service=lambda d: ~d["category"].isin(exc_func_for_service),
        )
        .groupby("id")
        .any()
        .loc[:, ["is_garage", "is_residential", "is_vacant", "is_service"]]
    )

    return df_short.join(data_long_features, validate="one_to_one").assign(
        type=lambda d: np.select(feature_types(d, high_rise), feature_names, "other"),
        time_period=lambda d: pd.cut(
            d["build_year_est"], bins, right=False, labels=labels
        ),
    )

In [11]:
build_features = add_building_features(data_gdynia, data_gdynia_long_categories)
pd.crosstab(
    index=build_features["time_period"],
    columns=build_features["type"],
    normalize="columns",
).style.background_gradient(cmap="Blues", axis="index").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)

type,garage,high-rise,mixed-use,other,residential,service
time_period,,,,,,
before1920,,,,,,0.8%
1920-39,0.9%,,49.1%,11.2%,16.1%,23.2%
1940-59,,24.1%,43.5%,21.2%,61.0%,23.2%
1960-89,95.3%,51.7%,,56.2%,16.1%,29.5%
1990+,3.8%,24.1%,7.5%,11.2%,6.8%,23.2%


4.

In [12]:
def street_metrics(df_long, df_short=None, min_functional=10):

    streets_long = pd.concat(
        [
            df_long.loc[lambda d: d["category"] != "brak parteru"]
            .assign(is_vacant=lambda d: d["category"] == "pustostan")
            .groupby("addr_street")
            .agg(num_premises=("category", "size"), num_vacant=("is_vacant", "sum"))
            .assign(
                num_premises_wo_vacant=lambda d: d["num_premises"] - d["num_vacant"],
                share_vacant=lambda d: d["num_vacant"].div(d["num_premises"]),
            ),
            df_long.loc[lambda d: ~d["category"].isin(["brak parteru", "pustostan"])]
            .groupby("addr_street")["category"]
            .value_counts()
            .to_frame("premise_per_cat")
            .assign(
                total_premises_per_street=lambda d: d.groupby("addr_street")[
                    "premise_per_cat"
                ].transform("sum"),
                share=lambda d: d["premise_per_cat"].div(
                    d["total_premises_per_street"]
                ),
                lnshare_mult_share=lambda d: np.log(d["share"]).mul(d["share"]).mul(-1),
            )
            .groupby("addr_street")
            .agg(
                num_cat=("premise_per_cat", lambda s: s.gt(0).sum()),
                leader_share=("share", "max"),
                func_entropy=("lnshare_mult_share", "sum"),
            )
            .assign(eff_cat=lambda d: np.exp(d["func_entropy"])),
        ],
        axis=1,
    ).loc[
        :,
        [
            "num_premises",
            "num_premises_wo_vacant",
            "num_vacant",
            "share_vacant",
            "leader_share",
            "func_entropy",
            "num_cat",
            "eff_cat",
        ],
    ]
    if df_short is not None:
        streets_long = streets_long.join(
            df_short.groupby("addr_street")
            .agg(
                num_buildings=("osm_id", "size"),
                num_mix_use=("type", lambda d: (d == "mixed-use").sum()),
            )
            .assign(share_mix_use=lambda d: d["num_mix_use"].div(d["num_buildings"]))
        )
    return streets_long.loc[lambda d: d["num_premises_wo_vacant"].ge(min_functional)]


In [ ]:
old_ids = data_gdynia.loc[
    lambda e: e["date_of_checking"] < pd.to_datetime("2026-10-22")
].index
data_gdynia_long_categories_before = data_gdynia_long_categories.loc[
    lambda d: d.index.isin(old_ids)
]


assert np.isclose(
    street_metrics(data_gdynia_long_categories_before, min_functional=0).loc[
        "Arkadiusza Rybickiego", "func_entropy"
    ],
    1.581,
    atol=0.001,
)
assert len(street_metrics(data_gdynia_long_categories_before, min_functional=10)) == 26

In [ ]:
data_gdynia_features = add_building_features(data_gdynia, data_gdynia_long_categories)
metrics_10 = street_metrics(
    data_gdynia_long_categories, data_gdynia_features, min_functional=10
)

assert len(metrics_10) == 28
metrics_10.sort_values("func_entropy", ascending=False).head(2)

,num_premises,num_premises_wo_vacant,num_vacant,share_vacant,leader_share,func_entropy,num_cat,eff_cat,num_buildings,num_mix_use,share_mix_use
addr_street,,,,,,,,,,,
Armii Krajowej,33,32,1,0.030303,0.28125,2.070699,10,7.930361,25,9,0.360000
3 Maja,29,28,1,0.034483,0.25000,2.044535,9,7.725564,21,9,0.428571


In [ ]:
street_metrics(
    data_gdynia_long_categories, data_gdynia_features, min_functional=10
).loc[
    :,
    [
        "num_premises_wo_vacant",
        "func_entropy",
        "share_vacant",
        "share_mix_use",
        "leader_share",
    ],
].corr(method="spearman")


,num_premises_wo_vacant,func_entropy,share_vacant,share_mix_use,leader_share
num_premises_wo_vacant,1.000000,0.304932,0.479918,0.124520,0.053353
func_entropy,0.304932,1.000000,-0.056164,0.215579,-0.771613
share_vacant,0.479918,-0.056164,1.000000,-0.023287,0.177264
share_mix_use,0.124520,0.215579,-0.023287,1.000000,-0.247426
leader_share,0.053353,-0.771613,0.177264,-0.247426,1.000000


In [ ]:
street_metrics(
    data_gdynia_long_categories, data_gdynia_features, min_functional=10
).loc[:, ["num_premises_wo_vacant", "func_entropy"]].corr(method="spearman")

,num_premises_wo_vacant,func_entropy
num_premises_wo_vacant,1.000000,0.304932
func_entropy,0.304932,1.000000


In [ ]:
street_metrics(
    data_gdynia_long_categories, data_gdynia_features, min_functional=10
).loc[
    lambda d: d.index != "Świętojańska", ["num_premises_wo_vacant", "func_entropy"]
].corr(method="pearson")

,num_premises_wo_vacant,func_entropy
num_premises_wo_vacant,1.000000,0.302331
func_entropy,0.302331,1.000000


In [ ]:
street_metrics(
    data_gdynia_long_categories, data_gdynia_features, min_functional=10
).loc[:, ["num_premises_wo_vacant", "func_entropy"]].corr(method="pearson")

,num_premises_wo_vacant,func_entropy
num_premises_wo_vacant,1.000000,0.095844
func_entropy,0.095844,1.000000


5.

In [ ]:
data_gdynia_features_old = add_building_features(
    data_gdynia.loc[lambda d: d.index.isin(old_ids)],
    data_gdynia_long_categories.loc[lambda d: d.index.isin(old_ids)],
)
entropy_before_after = pd.concat(
    {
        "before": street_metrics(
            data_gdynia_long_categories_before,
            data_gdynia_features_old,
            min_functional=10,
        )
        .loc[:, ["num_premises_wo_vacant", "func_entropy"]]
        .assign(
            rank_before=lambda d: d["func_entropy"].rank(
                method="dense", ascending=False
            )
        ),
        "after": street_metrics(
            data_gdynia_long_categories, data_gdynia_features, min_functional=10
        )
        .loc[:, ["num_premises_wo_vacant", "func_entropy"]]
        .assign(
            rank_after=lambda d: d["func_entropy"].rank(method="dense", ascending=False)
        ),
    },
    axis=1,
)

In [20]:
entropy_before_after

before               \
                                      num_premises_wo_vacant func_entropy   
addr_street                                                                 
10 Lutego                                               50.0     1.729192   
3 Maja                                                  28.0     2.044535   
Aleja Jana Pawła II                                     20.0     1.461636   
Antoniego Abrahama                                     112.0     1.869300   
Antoniego Hryniewickiego                                23.0     1.122093   
Armii Krajowej                                          32.0     2.070699   
Franciszka Żwirki i Stanisława Wigury                   18.0     1.618849   
Generała Józefa Bema                                    13.0     1.524707   
Jana Kilińskiego                                        19.0     1.763735   
Jana z Kolna                                            21.0     1.699326   
Jerzego Waszyngtona                                     25.0     1.754710   
Kazimierza Pułaskiego                                   13.0     1.412745   
Mściwoja                                                13.0     1.844621   
Obrońców Wybrzeża                                       11.0     1.240684   
Plac Kaszubski                                          17.0     1.564370   
Plac Konstytucji                                        15.0     1.436829   
Portowa                                                 16.0     1.894689   
Skwer Tadeusza Kościuszki                               19.0     1.556718   
Starowiejska                                            84.0     1.590618   
Stefana Batorego                                        12.0     1.791759   
Stefana Żeromskiego                                     32.0     1.579152   
Wójta Radtkego                                          40.0     1.810340   
Władysława IV                                           65.0     1.867471   
Zgoda                                                   12.0     1.473502   
Świętego Piotra                                         10.0     1.695743   
Świętojańska                                           214.0     1.522571   
Juliusza Słowackiego                                     NaN          NaN   
Żołnierzy I Armii Wojska Polskiego                       NaN          NaN   

                                                                   after  \
                                      rank_before num_premises_wo_vacant   
addr_street                                                                
10 Lutego                                    11.0                     50   
3 Maja                                        2.0                     28   
Aleja Jana Pawła II                          22.0                     20   
Antoniego Abrahama                            4.0                    112   
Antoniego Hryniewickiego                     26.0                     23   
Armii Krajowej                                1.0                     32   
Franciszka Żwirki i Stanisława Wigury        14.0                     18   
Generała Józefa Bema                         19.0                     25   
Jana Kilińskiego                              9.0                     22   
Jana z Kolna                                 12.0                     21   
Jerzego Waszyngtona                          10.0                     25   
Kazimierza Pułaskiego                        24.0                     13   
Mściwoja                                      6.0                     13   
Obrońców Wybrzeża                            25.0                     11   
Plac Kaszubski                               17.0                     17   
Plac Konstytucji                             23.0                     15   
Portowa                                       3.0                     16   
Skwer Tadeusza Kościuszki                    18.0                     19   
Starowiejska                                 15.0                     

In [ ]:
entropy_difference_before_after = (
    entropy_before_after.dropna()
    .sort_values(("after", "rank_after"))
    .assign(
        diff_rank=lambda d: d[("before", "rank_before")] != d[("after", "rank_after")]
    )
)

In [ ]:
entropy_difference_before_after.loc[
    :, ("before", ["num_premises_wo_vacant", "func_entropy"])
].corr(method="spearman")

before             
                              num_premises_wo_vacant func_entropy
before num_premises_wo_vacant               1.000000     0.329968
       func_entropy                         0.329968     1.000000

In [ ]:
entropy_difference_before_after.loc[
    :, ("after", ["num_premises_wo_vacant", "func_entropy"])
].corr(method="spearman")

after             
                             num_premises_wo_vacant func_entropy
after num_premises_wo_vacant               1.000000     0.307219
      func_entropy                         0.307219     1.000000

In [ ]:
entropy_difference_before_after.loc[
    :, [("after", "func_entropy"), ("before", "func_entropy")]
].corr(method="spearman")

,,after,before
,,func_entropy,func_entropy
after,func_entropy,1.00000,0.98906
before,func_entropy,0.98906,1.00000


6.

A.

In [25]:
tab1 = add_building_features(data_gdynia, data_gdynia_long_categories)

In [ ]:
pd.crosstab(
    index=tab1["time_period"], columns=tab1["is_vacant"], normalize="index"
).style.background_gradient(cmap="Blues", axis="columns").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)

is_vacant,False,True
time_period,,
before1920,100.0%,
1920-39,90.3%,9.7%
1940-59,96.3%,3.7%
1960-89,97.1%,2.9%
1990+,95.7%,4.3%


B.

In [ ]:
tab2 = tab1.loc[lambda d: d["is_vacant"] | d["is_service"]]
pd.crosstab(
    index=tab2["time_period"], columns=tab2["is_vacant"], normalize="index"
).style.background_gradient(cmap="Blues", axis="columns").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)

is_vacant,False,True
time_period,,
before1920,100.0%,
1920-39,89.4%,10.6%
1940-59,94.8%,5.2%
1960-89,87.8%,12.2%
1990+,94.6%,5.4%


C.

In [ ]:
tab3 = data_gdynia_long_categories.loc[lambda d: d["category"] != "brak parteru"].join(
    add_building_features(data_gdynia, data_gdynia_long_categories).loc[
        :, "time_period"
    ],
    validate="many_to_one",
)
pd.crosstab(
    index=tab3.reset_index().loc[:, "time_period"],
    columns=tab3.reset_index()["category"].eq("pustostan"),
    normalize="index",
).style.background_gradient(cmap="Blues", axis="columns").format(
    lambda v: "" if v == 0 else f"{v:.1%}"
)

category,False,True
time_period,,
before1920,100.0%,
1920-39,94.4%,5.6%
1940-59,97.3%,2.7%
1960-89,97.2%,2.8%
1990+,96.7%,3.3%
